# Coreset 选择方法实证研究 - 完整实验指南

本 Notebook 提供了一个完整的实验流程，用于测试和比较不同的 Coreset 选择方法。

## 实验内容

### 实验 1: 数据摘要 (Data Summarization)
比较不同 Coreset 方法在数据选择上的效果：
- Uniform (均匀随机采样)
- K-Center (K-中心聚类)
- Herding (核牧群方法)
- BCSR (双层优化的 Coreset 选择)

### 实验 2: 持续学习 (Continual Learning)
测试 Coreset 在持续学习中的效果：
- 任务增量学习场景
- 经验回放缓冲区
- 不同样本选择策略的比较

## 使用说明

1. **运行设置单元格**：克隆项目并安装依赖
2. **选择实验**：可以按顺序运行所有实验，或只运行特定实验
3. **调整参数**：每个实验都有可配置的参数
4. **查看结果**：实验完成后会自动生成可视化结果

## 注意事项

- 建议使用 GPU 运行时以加快训练速度
- 完整实验可能需要 30-60 分钟
- 可以减少 `NUM_EPOCHS` 或 `SELECTION_RATIO` 以快速测试

## 已知问题修复

✅ **Bug 1 修复**: 自动卸载 torchaudio 避免版本冲突
✅ **Bug 2 修复**: 使用正确的方法名 `uniform` 而非 `random`

---

## 快速开始指南

### 第一次使用？

1. **修改 GitHub 仓库地址**（如果使用 Colab）
   - 在下方「0.1 克隆项目仓库」单元格中
   - 将 `YOUR_USERNAME` 替换为你的 GitHub 用户名

2. **按顺序运行单元格**
   - 点击每个单元格左侧的 ▶️ 运行按钮
   - 或使用「运行时」→「全部运行」

3. **遇到依赖冲突？**
   - 查看「0.1.5 处理 Colab 预装包冲突」单元格
   - 设置 `FIX_CONFLICTS = True` 然后运行
   - 或者直接运行「0.3 安装依赖包」，系统会自动处理

### 常见问题

**Q: 提示 torchaudio/tensorflow 版本冲突？**

A: 这些包不是项目必需的。系统会自动卸载 torchaudio：
   - 运行「0.3 安装依赖包」会自动处理
   - 或手动运行「0.1.5」单元格设置 `FIX_CONFLICTS = True`

**Q: 提示 'Unknown method: random'？**

A: 已修复！现在使用正确的方法名：
   - ~~`random`~~ → **`uniform`** (均匀随机采样)
   - 其他方法：`kcenter`, `herding`, `bcsr`

**Q: scikit-learn/scipy 版本警告？**

A: 项目会自动安装兼容版本。如果仍有问题：
   - 重启 Colab 内核
   - 重新运行设置单元格

**Q: 如何减少实验时间？**

A: 在「0.4 设置全局参数」中调整：
   - `NUM_EPOCHS = 10`（默认 20）
   - `SELECTION_RATIO = 0.05`（默认 0.1）

---

## 项目概述

本 notebook 实现了以下 Coreset 选择方法：

| 方法 | 类型 | 说明 |
|------|------|------|
| Uniform | 基线 | 均匀随机采样 |
| K-Center | 基线 | 最大最小距离聚类 |
| Herding | 基线 | 核牧群方法 |
| BCSR | 提出方法 | 双层优化选择 |

持续学习实验支持多种缓冲区管理策略。

---

## 实验流程

```
第 0 步: 环境设置
  ├─ 克隆项目
  ├─ 处理依赖冲突（可选）
  ├─ 安装依赖（自动卸载 torchaudio）
  ├─ 导入模块
  └─ 设置参数

实验 1: 数据摘要
  ├─ 加载数据集
  ├─ 运行 4 种 Coreset 方法
  ├─ 训练和评估
  └─ 结果可视化

实验 2: 持续学习
  ├─ 创建任务数据集
  ├─ 逐任务训练
  ├─ 经验回放
  └─ 遗忘分析

实验总结
  ├─ 汇总所有结果
  ├─ 生成图表
  └─ 下载文件
```

---

---

# 第 0 步: 环境设置

运行此单元格以克隆项目并安装所有依赖。

In [ ]:
#@title 0.1 克隆项目仓库

import os
import sys
from pathlib import Path

# 检测是否在 Colab 中
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("✓ 检测到 Google Colab 环境")
    
    # 配置：替换为你的 GitHub 仓库地址
    GITHUB_REPO = "https://github.com/YOUR_USERNAME/coreset_benchmark.git"  #@param {type:"string"}
    
    if "YOUR_USERNAME" in GITHUB_REPO:
        print("\n⚠️ 请先修改 GITHUB_REPO 变量为你的实际仓库地址！")
        print("示例: https://github.com/username/coreset_benchmark.git")
    else:
        # 克隆项目
        PROJECT_DIR = Path("/content/coreset_benchmark")
        
        if not PROJECT_DIR.exists():
            print(f"正在克隆仓库: {GITHUB_REPO}")
            !git clone {GITHUB_REPO} {PROJECT_DIR}
            print("✓ 克隆完成")
        else:
            print(f"✓ 项目已存在: {PROJECT_DIR}")
        
        # 添加到 Python 路径
        if str(PROJECT_DIR) not in sys.path:
            sys.path.insert(0, str(PROJECT_DIR))
            print(f"✓ 已添加项目到 Python 路径")
        
        # 进入项目目录
        %cd {PROJECT_DIR}
        
else:
    # 本地环境
    print("✓ 本地环境")
    PROJECT_DIR = Path.cwd().parent
    if str(PROJECT_DIR) not in sys.path:
        sys.path.insert(0, str(PROJECT_DIR))

print(f"\n项目根目录: {PROJECT_DIR}")

In [ ]:
#@title 0.1.5 处理 Colab 预装包冲突（可选）

#@markdown 在 Colab 中，某些包可能预先安装但版本不兼容。
#@markdown 运行此单元格可以尝试修复已知的依赖冲突。

import subprocess
import sys

FIX_CONFLICTS = False  #@param {type:"boolean"}

if IN_COLAB and FIX_CONFLICTS:
    print("正在处理已知冲突...")
    
    # 卸载可能冲突的包（这些不是项目必需的）
    conflicting_packages = [
        "torchaudio",  # 项目不需要，但会与 torch 版本冲突
        "tensorflow",  # 项目不需要，但会与 tensorboard 版本冲突
    ]
    
    for pkg in conflicting_packages:
        try:
            subprocess.run(
                [sys.executable, "-m", "pip", "uninstall", "-y", pkg],
                capture_output=True
            )
            print(f"  ✓ 已卸载 {pkg}")
        except:
            pass
    
    # 升级关键包到兼容版本
    upgrades = [
        ("scipy", ">=1.14,<2.0"),
        ("scikit-learn", ">=1.6,<2.0"),
    ]
    
    for pkg, spec in upgrades:
        try:
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "-q", f"{pkg}{spec}"],
                capture_output=True
            )
            print(f"  ✓ 已升级 {pkg}")
        except:
            print(f"  ⚠️ {pkg} 升级失败")
    
    print("\n✓ 冲突处理完成")
    
    # 重启内核建议
    print("\n⚠️ 建议重启内核以使更改生效：")
    print("   点击 '运行时' -> '重启会话'")
    
else:
    print("跳过冲突处理")

In [ ]:
#@title 0.3 安装依赖包

import subprocess
import sys

def install_package_with_check(package_spec):
    """
    安装包并处理可能的冲突
    
    参数:
        package_spec: 包规格，如 'numpy>=1.24,<2.1'
    """
    try:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", package_spec
        ])
        return True
    except subprocess.CalledProcessError as e:
        print(f"⚠️ {package_spec} 安装失败: {e}")
        return False

if IN_COLAB:
    print("正在安装依赖包...")
    print("-" * 60)
    
    # 🔧 修复 Bug 1: 主动卸载冲突的 torchaudio
    print("\n处理预装包冲突...")
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "uninstall", "-y", "torchaudio"],
            capture_output=True
        )
        print("✓ 已卸载 torchaudio（避免与 torch 版本冲突）")
    except:
        print("  (torchaudio 未安装或卸载失败，继续)")
    
    # 按优先级顺序安装核心依赖
    # 分开安装以避免版本冲突
    
    core_packages = [
        "numpy>=1.24,<2.1",
        "pandas>=2.0,<2.5",
        "scipy>=1.14,<2.0",  # 更新以支持新版本
        "scikit-learn>=1.6,<2.0",  # 更新以支持新版本
        "matplotlib>=3.7,<3.10",
        "seaborn>=0.12,<0.14",
        "tqdm>=4.60",
        "pyyaml>=6.0",
    ]
    
    pytorch_packages = [
        "torch>=2.0,<2.6",
        "torchvision>=0.15,<0.21",
    ]
    
    # 安装核心包
    print("\n安装核心科学计算库...")
    for pkg in core_packages:
        install_package_with_check(pkg)
    
    # 安装 PyTorch (特殊处理以避免 torchaudio 冲突)
    print("\n安装 PyTorch...")
    for pkg in pytorch_packages:
        install_package_with_check(pkg)
    
    # 安装可选包 (tensorboard)
    print("\n安装可选依赖...")
    install_package_with_check("tensorboard>=2.13,<3.0")
    
    # 处理特定包的兼容性问题
    print("\n处理版本兼容性...")
    
    # 如果 scikit-learn 安装失败，尝试降级
    try:
        import sklearn
        print(f"✓ scikit-learn 版本: {sklearn.__version__}")
    except ImportError:
        print("⚠️ scikit-learn 安装失败，尝试安装兼容版本...")
        install_package_with_check("scikit-learn>=1.3,<2.0")
    
    print("-" * 60)
    print("✓ 依赖安装完成")
    
    # 检查关键包
    print("\n验证关键包安装:")
    try:
        import numpy, scipy, sklearn, torch, matplotlib
        print(f"  numpy: {numpy.__version__} ✓")
        print(f"  scipy: {scipy.__version__} ✓")
        print(f"  scikit-learn: {sklearn.__version__} ✓")
        print(f"  torch: {torch.__version__} ✓")
        print(f"  matplotlib: {matplotlib.__version__} ✓")
        
        # 验证 torchaudio 已卸载
        try:
            import torchaudio
            print(f"  ⚠️ torchaudio: {torchaudio.__version__} (仍存在，可能有版本冲突)")
        except ImportError:
            print(f"  ✓ torchaudio: 已卸载（无冲突）")
            
    except ImportError as e:
        print(f"⚠️ 部分包导入失败: {e}")
        
else:
    print("本地环境 - 跳过依赖安装")
    print("请确保已手动安装 requirements.txt 中的依赖")

# 导入基础库
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import time
import json
from datetime import datetime
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print(f"\n✓ 库导入完成")
print(f"  PyTorch 版本: {torch.__version__}")
print(f"  CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU 设备: {torch.cuda.get_device_name(0)}")

# 设置设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"  使用设备: {device}")

In [ ]:
#@title 0.3 导入项目模块

# 数据集
from src.datasets.data_loaders import (
    get_dataset,
    get_dataloader,
    DATASET_STATS
)

# 模型
from src.models.cnn import CNN_MNIST, CNN_CIFAR
from src.models.resnet import ResNet18
from src.models import get_model

# Coreset 方法
from src.baselines.baseline_methods import get_baseline
from src.coreset.bcsr_coreset import BCSRCoreset
from src.coreset.bilevel_coreset import BilevelCoreset

# 配置
from src.configs import (
    ExperimentConfig,
    BilevelConfig,
    CSReLConfig,
    DataSummarizationConfig,
    ContinualLearningConfig
)

print("✓ 项目模块导入完成")

In [ ]:
#@title 0.4 设置全局参数

#@markdown 数据集选择
DATASET = "MNIST"  #@param ["MNIST", "CIFAR10", "CIFAR100"]

#@markdown 训练参数
NUM_EPOCHS = 20  #@param {type:"integer"}
BATCH_SIZE = 128  #@param {type:"integer"}
LEARNING_RATE = 0.001  #@param {type:"number"}

#@markdown Coreset 参数
SELECTION_RATIO = 0.1  #@param {type:"slider", min:0.01, max:0.5, step:0.01}

#@markdown 其他参数
RANDOM_SEED = 42  #@param {type:"integer"}

# 设置随机种子
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# 创建结果目录
RESULTS_DIR = PROJECT_DIR / "results" / f"experiment_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("="*60)
print("实验配置")
print("="*60)
print(f"数据集: {DATASET}")
print(f"训练轮数: {NUM_EPOCHS}")
print(f"批次大小: {BATCH_SIZE}")
print(f"学习率: {LEARNING_RATE}")
print(f"选择比例: {SELECTION_RATIO*100}%")
print(f"设备: {device}")
print(f"随机种子: {RANDOM_SEED}")
print(f"结果保存路径: {RESULTS_DIR}")
print("="*60)

# 获取数据集统计信息
dataset_stats = DATASET_STATS[DATASET]
num_classes = dataset_stats['num_classes']
num_channels = dataset_stats['num_channels']
img_size = dataset_stats['img_size']

print(f"\n数据集信息:")
print(f"  类别数: {num_classes}")
print(f"  通道数: {num_channels}")
print(f"  图像大小: {img_size}x{img_size}")

---

# 实验 1: 数据摘要 (Data Summarization)

本实验比较不同 Coreset 选择方法的效果。

## 测试方法：
1. **Uniform**: 均匀随机采样
2. **K-Center**: K-中心聚类
3. **Herding**: 核牧群方法
4. **BCSR**: 双层优化的 Coreset 选择

## 评估指标：
- 选择时间
- Coreset 训练准确率
- 完整训练集准确率（对比）
- 性能下降百分比

In [ ]:
#@title 1.1 加载数据集

print("正在加载数据集...")

# 加载训练集和测试集
train_dataset = get_dataset(DATASET, train=True, download=True)
test_dataset = get_dataset(DATASET, train=False, download=True)

# 创建数据加载器
train_loader = get_dataloader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = get_dataloader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"✓ 数据集加载完成")
print(f"  训练集大小: {len(train_dataset)}")
print(f"  测试集大小: {len(test_dataset)}")

# 计算 Coreset 大小
coreset_size = int(len(train_dataset) * SELECTION_RATIO)
print(f"  Coreset 目标大小: {coreset_size} ({SELECTION_RATIO*100:.1f}%)")

In [ ]:
#@title 1.2 定义训练和评估函数

def train_model(model, train_loader, num_epochs, device, learning_rate=LEARNING_RATE):
    """训练模型"""
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    history = {'loss': [], 'acc': []}
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')
        for inputs, targets in pbar:
            inputs, targets = inputs.to(device), targets.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
            
            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100.*correct/total:.2f}%'})
        
        epoch_loss = running_loss / len(train_loader)
        epoch_acc = 100. * correct / total
        history['loss'].append(epoch_loss)
        history['acc'].append(epoch_acc)
    
    return history

def evaluate_model(model, test_loader, device):
    """评估模型"""
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    
    return 100. * correct / total

print("✓ 训练和评估函数定义完成")

In [ ]:
#@title 1.3 定义 Coreset 选择函数

def apply_coreset_method(method_name, train_dataset, coreset_size, device):
    """
    应用指定的 Coreset 选择方法
    
    参数:
        method_name: 方法名称 ('uniform', 'kcenter', 'herding', 'bcsr')
        train_dataset: 训练数据集
        coreset_size: Coreset 大小
        device: 计算设备
    
    返回:
        selected_indices: 选中的样本索引
        selection_time: 选择耗时
    """
    print(f"\n{'='*60}")
    print(f"应用 {method_name.upper()} 方法")
    print(f"{'='*60}")
    
    # 提取所有特征
    print("提取特征...")
    all_features = []
    all_labels = []
    for inputs, labels in train_loader:
        all_features.append(inputs.view(inputs.size(0), -1))
        all_labels.append(labels)
    
    X_train = torch.cat(all_features, dim=0)
    y_train = torch.cat(all_labels, dim=0)
    print(f"特征形状: {X_train.shape}")
    
    # 开始选择
    start_time = time.time()
    
    if method_name == 'bcsr':
        # BCSR 方法
        X_train_dev = X_train.to(device)
        y_train_dev = y_train.to(device)
        
        selector = BCSRCoreset(
            learning_rate_inner=0.01,
            learning_rate_outer=0.1,
            num_inner_steps=50,
            num_outer_steps=20,
            device=device,
            random_state=RANDOM_SEED
        )
        
        _, _, info = selector.coreset_select(
            X=X_train_dev,
            y=y_train_dev,
            coreset_size=coreset_size,
            model=None  # 使用简化核方法
        )
        selected_indices = info['selected_indices']
    
    else:
        # 基线方法（注意：方法名必须是 'uniform', 'kcenter', 'herding' 等）
        X_np = X_train.numpy()
        y_np = y_train.numpy()
        
        baseline = get_baseline(method_name)
        
        if method_name == 'herding':
            selected_indices = baseline.select(
                X_np, y_np, size=coreset_size,
                kernel='rbf', gamma=0.1, random_state=RANDOM_SEED
            )
        else:
            selected_indices = baseline.select(
                X_np, y_np, size=coreset_size,
                random_state=RANDOM_SEED
            )
    
    selection_time = time.time() - start_time
    
    print(f"✓ 选择完成")
    print(f"  选中的样本数: {len(selected_indices)}")
    print(f"  选择耗时: {selection_time:.2f}秒")
    
    return selected_indices, selection_time

print("✓ Coreset 选择函数定义完成")

In [ ]:
#@title 1.4 运行所有 Coreset 方法对比

# 要测试的方法（🔧 修复 Bug 2: 使用正确的方法名）
METHODS = ['uniform', 'kcenter', 'herding', 'bcsr']

# 存储结果
results_summary = []

# 创建模型工厂函数
if DATASET == 'MNIST':
    model_factory = lambda: CNN_MNIST(num_classes=num_classes)
else:
    model_factory = lambda: ResNet18(num_classes=num_classes)

# 对每种方法运行实验
for method in METHODS:
    print(f"\n\n{'#'*80}")
    print(f"# 实验: {method.upper()}")
    print(f"{'#'*80}")
    
    # 应用 Coreset 选择
    selected_indices, selection_time = apply_coreset_method(
        method, train_dataset, coreset_size, device
    )
    
    # 创建 Coreset 数据集
    from torch.utils.data import Subset
    coreset_dataset = Subset(train_dataset, selected_indices)
    coreset_loader = get_dataloader(coreset_dataset, batch_size=BATCH_SIZE, shuffle=True)
    
    # 在 Coreset 上训练
    print(f"\n在 Coreset 上训练模型...")
    model = model_factory()
    history = train_model(model, coreset_loader, NUM_EPOCHS, device)
    
    # 评估
    train_acc = history['acc'][-1]
    test_acc = evaluate_model(model, test_loader, device)
    
    print(f"\n结果:")
    print(f"  训练准确率: {train_acc:.2f}%")
    print(f"  测试准确率: {test_acc:.2f}%")
    
    # 保存结果
    results_summary.append({
        'method': method,
        'coreset_size': len(selected_indices),
        'selection_time': selection_time,
        'train_acc': train_acc,
        'test_acc': test_acc,
        'history': history
    })

print(f"\n\n{'='*80}")
print(f"所有实验完成!")
print(f"{'='*80}")

In [ ]:
#@title 1.5 完整训练集对比（可选）

#@markdown 运行此单元格可以训练一个在完整训练集上的模型作为对比基准
#@markdown 注意：这将需要额外的时间

RUN_FULL_TRAINING = False  #@param {type:"boolean"}

if RUN_FULL_TRAINING:
    print(f"\n{'='*60}")
    print(f"在完整训练集上训练")
    print(f"{'='*60}")
    
    model_full = model_factory()
    history_full = train_model(model_full, train_loader, NUM_EPOCHS, device)
    test_acc_full = evaluate_model(model_full, test_loader, device)
    
    print(f"\n完整训练集结果:")
    print(f"  测试准确率: {test_acc_full:.2f}%")
    
    # 更新结果
    for result in results_summary:
        result['test_acc_full'] = test_acc_full
        result['performance_drop'] = test_acc_full - result['test_acc']
else:
    print("跳过完整训练集对比")

In [ ]:
#@title 1.6 结果可视化

# 创建图表
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 提取数据
methods = [r['method'] for r in results_summary]
test_accs = [r['test_acc'] for r in results_summary]
selection_times = [r['selection_time'] for r in results_summary]

# 1. 测试准确率对比
axes[0, 0].bar(methods, test_accs, color='steelblue', alpha=0.7)
axes[0, 0].set_ylabel('准确率 (%)')
axes[0, 0].set_title('测试准确率对比')
axes[0, 0].set_ylim([min(test_accs) - 5, 100])
axes[0, 0].grid(axis='y', alpha=0.3)
for i, v in enumerate(test_accs):
    axes[0, 0].text(i, v + 0.5, f'{v:.2f}%', ha='center')

# 2. 选择时间对比
axes[0, 1].bar(methods, selection_times, color='coral', alpha=0.7)
axes[0, 1].set_ylabel('时间 (秒)')
axes[0, 1].set_title('选择时间对比')
axes[0, 1].grid(axis='y', alpha=0.3)
for i, v in enumerate(selection_times):
    axes[0, 1].text(i, v + 0.5, f'{v:.2f}s', ha='center')

# 3. 训练曲线
colors = ['blue', 'green', 'orange', 'red']
for i, result in enumerate(results_summary):
    axes[1, 0].plot(result['history']['acc'], label=result['method'].upper(), 
                   color=colors[i], linewidth=2)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('训练准确率 (%)')
axes[1, 0].set_title('训练曲线')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# 4. 损失曲线
for i, result in enumerate(results_summary):
    axes[1, 1].plot(result['history']['loss'], label=result['method'].upper(),
                   color=colors[i], linewidth=2)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('训练损失')
axes[1, 1].set_title('损失曲线')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'experiment1_comparison.png', dpi=150)
plt.show()

print("✓ 可视化图表已保存")

In [ ]:
#@title 1.7 结果汇总表

import pandas as pd

# 创建结果表格
df_results = pd.DataFrame(results_summary)
df_results = df_results[['method', 'coreset_size', 'selection_time', 'train_acc', 'test_acc']]
df_results.columns = ['方法', 'Coreset大小', '选择时间(秒)', '训练准确率(%)', '测试准确率(%)']

print("\n" + "="*80)
print("实验 1 结果汇总")
print("="*80)
print(df_results.to_string(index=False))
print("="*80)

# 保存结果
df_results.to_csv(RESULTS_DIR / 'experiment1_results.csv', index=False)
with open(RESULTS_DIR / 'experiment1_results.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

print(f"\n结果已保存到: {RESULTS_DIR}")

---

# 实验 2: 持续学习 (Continual Learning)

本实验测试 Coreset 在任务增量学习场景中的效果。

## 实验设置：
- 将 MNIST/CIFAR-10 的类别划分为多个任务
- 使用经验回放缓冲区存储 Coreset
- 依次学习每个任务并评估所有已见任务

## 评估指标：
- 平均准确率
- 遗忘度量

In [ ]:
#@title 2.1 持续学习实验参数

#@markdown 任务设置
NUM_TASKS = 5  #@param {type:"integer"}
NUM_CLASSES_PER_TASK = 2  #@param {type:"integer"}

#@markdown 缓冲区设置
MEMORY_SIZE = 2000  #@param {type:"integer"}
BUFFER_RATIO = 0.3  #@param {type:"slider", min:0.1, max:0.9, step:0.1}

#@markdown Coreset 选择方法（🔧 修复 Bug 2: 使用正确的方法名）
SELECTION_METHOD = "uniform"  #@param ["uniform", "loss", "margin", "gradient"]

#@markdown 训练参数
CL_NUM_EPOCHS = 10  #@param {type:"integer"}

print("持续学习实验配置：")
print(f"  任务数量: {NUM_TASKS}")
print(f"  每任务类别数: {NUM_CLASSES_PER_TASK}")
print(f"  缓冲区大小: {MEMORY_SIZE}")
print(f"  缓冲区损失权重: {BUFFER_RATIO}")
print(f"  选择方法: {SELECTION_METHOD}")

In [ ]:
#@title 2.2 导入持续学习模块

# 从实验脚本导入
import sys
sys.path.insert(0, str(PROJECT_DIR / "experiments"))

from continual_learning import (
    CoresetBuffer,
    train_task,
    evaluate_all_tasks,
    create_task_datasets,
    compute_average_accuracy,
    compute_forgetting_measure
)

print("✓ 持续学习模块导入完成")

In [ ]:
#@title 2.3 创建任务数据集

print(f"创建任务增量学习数据集...")

train_loaders, test_loaders, num_classes_cl, input_shape = create_task_datasets(
    dataset_name=DATASET,
    num_tasks=NUM_TASKS,
    num_classes_per_task=NUM_CLASSES_PER_TASK,
    batch_size=BATCH_SIZE,
    data_root=str(PROJECT_DIR / 'data')
)

print(f"\n✓ 任务数据集创建完成")
print(f"  总类别数: {num_classes_cl}")
print(f"  输入形状: {input_shape}")

In [ ]:
#@title 2.4 创建模型和缓冲区

# 创建模型
if DATASET == 'MNIST':
    model_cl = CNN_MNIST(num_classes=NUM_CLASSES_PER_TASK)
else:
    model_cl = ResNet18(num_classes=NUM_CLASSES_PER_TASK)

model_cl = model_cl.to(device)

# 创建缓冲区
buffer = CoresetBuffer(
    memory_size=MEMORY_SIZE,
    input_shape=input_shape,
    num_classes=num_classes_cl,
    device=str(device)
)

print(f"✓ 模型和缓冲区创建完成")
print(f"  模型参数量: {sum(p.numel() for p in model_cl.parameters()):,}")
print(f"  缓冲区容量: {MEMORY_SIZE}")

In [ ]:
#@title 2.5 运行持续学习实验

accuracy_matrix = np.zeros((NUM_TASKS, NUM_TASKS))

print(f"\n{'='*80}")
print(f"开始持续学习训练")
print(f"{'='*80}")

for task_id in range(NUM_TASKS):
    print(f"\n{'#'*60}")
    print(f"# 学习任务 {task_id}")
    print(f"{'#'*60}")
    
    # 训练当前任务
    train_stats = train_task(
        model=model_cl,
        train_loader=train_loaders[task_id],
        buffer=buffer,
        task_id=task_id,
        num_epochs=CL_NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        device=device,
        buffer_ratio=BUFFER_RATIO,
        selection_method=SELECTION_METHOD
    )
    
    print(f"训练完成 - 损失: {train_stats['loss']:.4f}, 准确率: {train_stats['accuracy']:.2f}%")
    
    # 从当前任务选择样本添加到缓冲区
    all_data = []
    all_labels = []
    for data, labels in train_loaders[task_id]:
        all_data.append(data)
        all_labels.append(labels)
    
    all_data = torch.cat(all_data, dim=0)
    all_labels = torch.cat(all_labels, dim=0)
    
    # 选择 Coreset 样本
    num_samples = min(MEMORY_SIZE // NUM_TASKS, len(all_data))
    selected_data, selected_labels = buffer.select_coreset(
        data=all_data,
        labels=all_labels,
        num_samples=num_samples,
        method=SELECTION_METHOD,
        model=model_cl
    )
    
    # 添加到缓冲区
    added = buffer.add(
        data=selected_data,
        labels=selected_labels,
        task_id=task_id,
        selection_method=SELECTION_METHOD
    )
    
    print(f"添加 {added} 个样本到缓冲区")
    print(f"缓冲区状态: {len(buffer.data)}/{buffer.memory_size}")
    
    # 在所有已见任务上评估
    print(f"\n在所有任务上评估:")
    accuracies = evaluate_all_tasks(model_cl, test_loaders, device)
    
    # 更新准确率矩阵
    accuracy_matrix[task_id, :task_id+1] = accuracies[:task_id+1]

print(f"\n{'='*80}")
print(f"持续学习实验完成!")
print(f"{'='*80}")

In [ ]:
#@title 2.6 结果可视化

# 计算指标
avg_accuracy = compute_average_accuracy(accuracy_matrix)
forgetting = compute_forgetting_measure(accuracy_matrix)

# 创建图表
fig = plt.figure(figsize=(16, 6))

# 1. 准确率矩阵热图
ax1 = plt.subplot(1, 3, 1)
im = ax1.imshow(accuracy_matrix, cmap='YlOrRd', aspect='auto', vmin=0, vmax=100)
ax1.set_xlabel('任务 ID')
ax1.set_ylabel('学习顺序')
ax1.set_title('准确率矩阵 (%)')

# 添加数值标注
for i in range(NUM_TASKS):
    for j in range(NUM_TASKS):
        if j <= i:
            text = ax1.text(j, i, f'{accuracy_matrix[i, j]:.1f}',
                          ha="center", va="center", color="black", fontsize=9)

plt.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)

# 2. 每个任务准确率变化
ax2 = plt.subplot(1, 3, 2)
for task_id in range(NUM_TASKS):
    # 绘制该任务在学习过程中的准确率变化
    valid_mask = np.arange(task_id, NUM_TASKS)
    ax2.plot(valid_mask, accuracy_matrix[valid_mask, task_id], 
            marker='o', label=f'Task {task_id}', linewidth=2)

ax2.set_xlabel('学习顺序')
ax2.set_ylabel('准确率 (%)')
ax2.set_title('各任务准确率变化')
ax2.legend(loc='upper right', fontsize=8)
ax2.grid(alpha=0.3)
ax2.set_ylim([0, 105])

# 3. 汇总指标
ax3 = plt.subplot(1, 3, 3)
ax3.axis('off')

# 最终准确率
final_accuracies = accuracy_matrix[-1, :]
summary_text = f"""
实验汇总
="*30 + "\n",
数据集: {DATASET}
任务数: {NUM_TASKS}
缓冲区: {MEMORY_SIZE}
选择方法: {SELECTION_METHOD}

性能指标:
{'─'*30}

平均准确率:
  {avg_accuracy:.2f}%

遗忘度量:
  {forgetting:.2f}%

各任务最终准确率:
"""

for i, acc in enumerate(final_accuracies):
    summary_text += f"  任务{i}: {acc:.2f}%\n"

ax3.text(0.1, 0.5, summary_text, transform=ax3.transAxes,
        fontsize=11, verticalalignment='center',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle(f'持续学习实验结果 - {DATASET}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'experiment2_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ 可视化图表已保存")

In [ ]:
#@title 2.7 保存结果

results_cl = {
    'dataset': DATASET,
    'num_tasks': NUM_TASKS,
    'num_classes_per_task': NUM_CLASSES_PER_TASK,
    'memory_size': MEMORY_SIZE,
    'selection_method': SELECTION_METHOD,
    'buffer_ratio': BUFFER_RATIO,
    'num_epochs': CL_NUM_EPOCHS,
    'learning_rate': LEARNING_RATE,
    'batch_size': BATCH_SIZE,
    'seed': RANDOM_SEED,
    'accuracy_matrix': accuracy_matrix.tolist(),
    'average_accuracy': float(avg_accuracy),
    'forgetting_measure': float(forgetting),
    'final_accuracies': final_accuracies.tolist()
}

# 保存 JSON
with open(RESULTS_DIR / 'experiment2_results.json', 'w') as f:
    json.dump(results_cl, f, indent=2)

# 保存 CSV
df_cl = pd.DataFrame(accuracy_matrix)
df_cl.columns = [f'Task_{i}' for i in range(NUM_TASKS)]
df_cl.index = [f'After_Task_{i}' for i in range(NUM_TASKS)]
df_cl.to_csv(RESULTS_DIR / 'experiment2_accuracy_matrix.csv')

print("\n" + "="*80)
print("实验 2 结果汇总")
print("="*80)
print(f"\n平均准确率: {avg_accuracy:.2f}%")
print(f"遗忘度量: {forgetting:.2f}%")
print(f"\n各任务最终准确率:")
for i, acc in enumerate(final_accuracies):
    print(f"  任务 {i}: {acc:.2f}%")
print("="*80)
print(f"\n结果已保存到: {RESULTS_DIR}")

---

# 实验总结

所有实验已完成！以下是结果总结。

In [ ]:
#@title 打印所有结果

print("\n" + "="*80)
print("全部实验总结")
print("="*80)

print(f"\n数据集: {DATASET}")
print(f"设备: {device}")
print(f"随机种子: {RANDOM_SEED}")
print(f"结果保存路径: {RESULTS_DIR}\n")

print("-" * 80)
print("实验 1: 数据摘要")
print("-" * 80)
for r in results_summary:
    print(f"  {r['method'].upper():8s}: 测试准确率={r['test_acc']:5.2f}%, 选择时间={r['selection_time']:5.2f}s")

print("\n" + "-" * 80)
print("实验 2: 持续学习")
print("-" * 80)
print(f"  平均准确率: {avg_accuracy:.2f}%")
print(f"  遗忘度量: {forgetting:.2f}%")

print("\n" + "="*80)
print(f"所有结果已保存到: {RESULTS_DIR}")
print("="*80)

In [ ]:
#@title 下载结果（Colab 专用）

if IN_COLAB:
    from google.colab import files
    
    print("正在压缩结果文件...")
    !zip -r {RESULTS_DIR.name}.zip {RESULTS_DIR} > /dev/null
    
    print(f"下载结果: {RESULTS_DIR.name}.zip")
    files.download(f"{RESULTS_DIR.name}.zip")
else:
    print("本地环境 - 结果已保存到磁盘")